# 🚀 pypff — Interactive Demo

`pypff` is a high-performance library for reading PanoSETI File Format (`.pff`) data.
All reads go through `mmap` so they are **zero-copy** by default.

| Feature | API |
|---------|-----|
| Run discovery | `PanosetiRun(path)` |
| Streaming frames | `seq.iter_batches(size=256)` |
| Random-access | `seq[i]`, `seq[0:100:2]`, `seq.read_images(indices)` |
| Nanosecond timestamps (batch) | `seq.timestamps_at(indices)` |
| Full timestamp cache | `seq.timestamps()` |
| Time navigation | `seq.seek_time(t_ns)` |
| Single-pass metadata | `seq.get_metadata_arrays(keys)` |
| Validated headers | `seq.get_frame_validated(i)` |
| Housekeeping telemetry | `run.get_hk()` |
| Time-multiplex analysis | see **§ 5** |

Jump to any section, or run all cells top-to-bottom for a full walkthrough.

In [ ]:
%matplotlib inline
import os
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import clear_output, display
from plotly.subplots import make_subplots
from tqdm.notebook import tqdm

from pypff import PanosetiRun, PFFSequence, hkpff

In [ ]:
# Set path to example data
DATA_DIR = Path("/mnt/beegfs/data/L0")

if not DATA_DIR.exists():
    print("❌ Example data directory not found. Please ensure you are in the pypff/example directory.")
else:
    print("".join([f'{d}\n' for d in DATA_DIR.iterdir()]))

## 1. Initialize the Run

Scanning the directory to discover available products and configuration metadata.

In [ ]:
obs = '/mnt/beegfs/data/L0/obs_Lick.start_2023-10-12T04:28:34Z.runtype_eng-test.pffd'
obs_path = DATA_DIR / obs

run = PanosetiRun(obs_path)
products = run.list_products()

print(f"✅ Loaded Run: {run.run_dir.name}")
print(f"Found {len(products)} Data Products.")
run.show()

In [ ]:
run.list_products()

In [ ]:
seq2 = run.get_product('dp_img16.bpp_2.module_1')
seq2.metadata_offsets

In [ ]:
# JSON header and image array
seq2.get_frame(10)

In [ ]:
# Pydantic-validated frame
seq2.get_frame_validated(10)

In [ ]:
# int64 ns-precision timestamp
seq2.timestamp_at(1)

## 2. Quick-Start Feature Tour

This section mirrors the **README quick-start** and demonstrates all major APIs
on live data.  Each sub-section is self-contained — feel free to run just the
cells you care about.

### 2a. Streaming frames (memory-bounded)

`iter_batches` yields **zero-copy strided views** into the mmap — the entire
file is never loaded into RAM.  The optional `with_timestamps=True` flag delivers
`int64` nanosecond timestamps alongside each batch in the same pass.

In [ ]:
seq = run.get_product(run.list_products()[0])

print(f"Product : {seq.name}")
print(f"Frames  : {len(seq):,}")
print(f"Shape   : {seq.frame_config.image_shape}")
print(f"dtype   : {seq.frame_config.dtype}")
print()

# Collect the first 3 batches (with timestamps) to show the API
batch_size = 64
for batch_idx, (imgs, ts) in enumerate(
    seq.iter_batches(size=batch_size, with_timestamps=True)
):
    print(
        f"  batch {batch_idx:02d} | frames {batch_idx*batch_size}–{batch_idx*batch_size + len(ts) - 1}"
        f" | shape {imgs.shape} | t[0]={ts[0]:,} ns"
    )
    if batch_idx == 2:
        print("  … (stopping early)")
        break

### 2b. Random access and slicing

`seq[i]` is a **zero-copy strided view** (no copy).
`seq[start:stop:step]` and `seq.read_images(indices)` both sort indices
internally for disk locality before reading.

In [ ]:
seq = run.get_product(run.list_products()[0])

# Single frame — zero-copy view
frame = seq[0]
print(f"seq[0]              shape={frame.shape}  dtype={frame.dtype}  sum={frame.sum()}")

# Slice  (every other frame, first 20)
sliced = seq[0:20:2]
print(f"seq[0:20:2]         shape={sliced.shape}")

# Unsorted random access — sorted internally for locality
indices = np.array([len(seq)-1, 0, len(seq)//2, 5])
imgs = seq.read_images(indices)
print(f"read_images([...])  shape={imgs.shape}")

# Negative indexing
last = seq[-1]
print(f"seq[-1] == seq[{len(seq)-1}]  equal={np.array_equal(last, seq[len(seq)-1])}")

### 2c. Nanosecond-precise timestamps

All timestamp APIs return **`int64` nanoseconds** — no float precision loss.
`timestamps_at(indices)` is the vectorized batch API; it issues one
`np.frombuffer` call per file group instead of a Python loop.

In [ ]:
seq = run.get_product(run.list_products()[0])

# Full cached array
ts_all = seq.timestamps()                     # int64 ns
print(f"timestamps()         dtype={ts_all.dtype}  len={len(ts_all):,}")

# Zero-copy datetime64[ns] view — works natively with matplotlib date axes
ts_dt = seq.timestamps(as_datetime=True)
print(f"timestamps(as_datetime=True)  dtype={ts_dt.dtype}")

# Single frame (convenience wrapper around timestamps_at)
t0 = seq.timestamp_at(0)
print(f"timestamp_at(0)      {t0:,} ns  ({t0/1e9:.3f} s since epoch)")

# Batch random access — vectorized, one frombuffer pass per file group
stride = max(len(seq) // 20, 1)
sample_idx = np.arange(0, len(seq), stride, dtype=np.int64)
ts_batch = seq.timestamps_at(sample_idx)      # shape (N,)  int64
print(f"timestamps_at({len(sample_idx)} samples)  dtype={ts_batch.dtype}  shape={ts_batch.shape}")

# Elapsed time in float seconds (subtract first in integer space to avoid precision loss)
elapsed_s = (ts_batch - ts_batch[0]) / 1e9
print(f"Run duration:        {elapsed_s[-1]:.3f} s  (from {len(sample_idx)} samples)")

# Time navigation — find the frame closest to 0.5 s into the run
target_ns = int(ts_all[0]) + int(0.5e9)
idx = seq.seek_time(target_ns)
print(f"seek_time(t0 + 0.5 s)  → frame {idx}")

### 2d. Single-pass metadata extraction

`get_metadata_arrays(keys)` builds a single composite `np.dtype` covering all
requested fields and issues **one `np.frombuffer` call per file** — regardless
of how many keys you request.  The virtual key `'unix_t_ns'` gives the same
nanosecond timestamps as `timestamps()`.

In [ ]:
seq = run.get_product(run.list_products()[0])

# Discover what header fields are available
print("Header fields:", list(seq.metadata_offsets.keys())[:10], "...")
print()

# Extract multiple fields in one pass
available_keys = list(seq.metadata_offsets.keys())[:4]
meta = seq.get_metadata_arrays(available_keys + ["unix_t_ns"])
for k, v in meta.items():
    print(f"  {k:30s}  dtype={v.dtype}  first={v[0]:,}")

print()

# Indexed path: extract only a subset of frames (also vectorized)
sample_idx = np.array([0, 1, len(seq)//2, len(seq)-1], dtype=np.int64)
meta_idx = seq.get_metadata_arrays(available_keys, indices=sample_idx)
print(f"Indexed extract for frames {list(sample_idx)}:")
for k, v in meta_idx.items():
    print(f"  {k:30s}  {list(v)}")

### 2e. Frame headers and Pydantic validation

`get_frame(i)` returns a raw `dict` (fastest).
`get_frame_validated(i)` returns a fully validated Pydantic model.

In [ ]:
seq = run.get_product(run.list_products()[0])

# Raw dict — fastest, no Pydantic overhead
header_dict, img = seq.get_frame(0)
print("Raw header type :", type(header_dict).__name__)
print("Image shape     :", img.shape, " dtype:", img.dtype)
print()

# Pydantic-validated — use for inspection, not tight loops
header_val, _ = seq.get_frame_validated(0)
print("Validated header:", type(header_val).__name__)
print("timestamp_ns    :", header_val.timestamp_ns)

## 3. Interactive Frame Explorer

Select a product and navigate through frames using the slider.

In [ ]:
product_dropdown = widgets.Dropdown(
    options=products,
    description='Product:',
    disabled=False,
)

frame_slider = widgets.IntSlider(
    min=0,
    max=100,
    step=1,
    description='Frame:',
    continuous_update=True,
    layout=widgets.Layout(width='500px')
)

out = widgets.Output()

def update_display(change=None):
    product_name = product_dropdown.value
    seq = run.get_product(product_name)
    frame_slider.max = len(seq) - 1
    
    frame_idx = frame_slider.value
    header, image = seq.get_frame_validated(frame_idx)
    
    with out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(8, 6))
        im = ax.imshow(image, cmap='magma', interpolation='nearest', origin='lower')
        fig.colorbar(im, ax=ax, label='ADC Value')
        
        # Handle different header types
        pkt_num = header.pkt_num if hasattr(header, 'pkt_num') else header.quabo_0.pkt_num
            
        ax.set_title(f"{product_name}\nFrame {frame_idx} | Packet {pkt_num} | {header.timestamp_ns} ns", loc='left')
        plt.show()

product_dropdown.observe(update_display, names='value')
frame_slider.observe(update_display, names='value')
update_display()

display(widgets.VBox([product_dropdown, frame_slider, out]))

## 4. Multi-Frame Grid Sample

Visualizing a sequence of frames from each product to inspect temporal variations.

In [ ]:
n_samples = 8
fig, axes = plt.subplots(len(products), n_samples, figsize=(2 * n_samples, 2.5 * len(products)), squeeze=False)

for row_idx, name in enumerate(products):
    seq = run.get_product(name)
    images = seq.read_images_range(start=0, count=n_samples)
    
    for col_idx in range(n_samples):
        ax = axes[row_idx, col_idx]
        ax.imshow(images[col_idx], cmap='viridis', origin='lower')
        ax.axis('off')
        if col_idx == 0:
            ax.set_title(f"{name}\n{seq.frame_config.image_shape}", loc='right', fontweight='bold', fontsize=9)
        if row_idx == 0:
            ax.set_title(f"T+{col_idx}", fontsize=10)

plt.tight_layout()
plt.show()

## 5. Interactive Housekeeping Monitor

Explore telemetry parameters from the housekeeping stream.

In [ ]:
hk_data = hkpff(obs_path / "hk.pff").readhk()

device_select = widgets.Select(options=sorted(hk_data.keys()), description='Device:')
param_select = widgets.Select(options=[], description='Param:')
hk_out = widgets.Output()

def update_params(change=None):
    device = device_select.value
    param_select.options = sorted(hk_data[device].keys())

def update_hk_plot(change=None):
    device = device_select.value
    param = param_select.value
    if not param:
        return
    
    values = hk_data[device][param]
    with hk_out:
        clear_output(wait=True)
        plt.figure(figsize=(10, 4))
        plt.plot(values, color='tab:blue', alpha=0.8)
        plt.title(f"{device} - {param}")
        plt.grid(True, alpha=0.3)
        plt.show()

device_select.observe(update_params, names='value')
device_select.observe(update_hk_plot, names='value')
param_select.observe(update_hk_plot, names='value')

update_params()
update_hk_plot()
display(widgets.HBox([device_select, param_select]), hk_out)

## 6. Time-Multiplexed Interleaving Analysis

PanoSETI often operates in an interleaved mode, switching between high-rate Pulse Height detection and standard Movie-mode imaging. 

The following visualization analyzes the collection rate and scheduling of all discovered data products simultaneously.

In [ ]:
def extract_timeline(run_obj, sample_rate=1000):
    data = {}
    for prod_name in tqdm(run_obj.list_products(), desc="Scanning Timestamps"):
        seq = run_obj.get_product(prod_name)
        if len(seq) == 0:
            continue
        
        # Build sample indices across the full sequence
        indices = np.arange(0, len(seq), sample_rate, dtype=np.int64)
        if len(indices) == 0 or indices[-1] != len(seq) - 1:
            indices = np.append(indices, len(seq) - 1)
        
        # Single vectorized call — one np.frombuffer pass per file, not one per index
        times = seq.timestamps_at(indices) / 1e9
        data[prod_name] = {"times": times, "indices": indices}
    return data

max_nbins = 1000
sample_rate = 1000
timeline = extract_timeline(run, sample_rate)

if timeline:
    # Normalize time to start of run
    t0 = min([d["times"].min() for d in timeline.values()])
    max_t = max([d["times"].max() for d in timeline.values()]) - t0
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
    
    # Float division so short runs don't collapse to bin_size=1
    bin_size = max(max_t / max_nbins, 1e-3)
    bins = np.arange(0, max_t + bin_size, bin_size)
    colors = plt.cm.tab10.colors
    
    for i, (name, d) in enumerate(timeline.items()):
        norm_times = d["times"] - t0
        counts, _ = np.histogram(norm_times, bins=bins)
        # Each sample represents `sample_rate` real frames;
        # divide by bin_size (seconds) to get frames/sec
        fps = (counts * sample_rate) / bin_size
        
        color = colors[i % len(colors)]
        
        # Rate Plot
        ax1.step(bins[:-1], fps, where='post', label=name, color=color, linewidth=2)
        ax1.fill_between(bins[:-1], fps, step='post', color=color, alpha=0.3)
        
        # Gantt Plot (Blocks indicate activity)
        active_bins = bins[:-1][fps > 0]
        ax2.scatter(active_bins, [name] * len(active_bins), marker='s', s=50, color=color, label=name)

    ax1.set_title('Collection Rate (FPS)', fontweight='bold')
    ax1.set_ylabel('FPS')
    ax1.legend(loc='upper right')
    ax1.grid(True, alpha=0.3)
    
    ax2.set_title('Time-Multiplex Schedule', fontweight='bold')
    ax2.set_xlabel('Seconds since Run Start')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()